# Week 5 – Milestone 1: Smart Logistics Tracking — Blockchain Ledger
**MO-IT148 | Smart Tracking System**

---
## SECTION 1: SETTING UP THE ENVIRONMENT

In [18]:
from web3 import Web3

# Connect to local Ganache blockchain
ganache_url = "http://127.0.0.1:7545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

print("========================================")
print("  SETTING UP THE ENVIRONMENT")
print("========================================")

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
    print(f"   Was your connection successful? YES")
    print(f"   Chain ID  : {web3.eth.chain_id}")
    print(f"   Block #   : {web3.eth.block_number}")
    print(f"   Accounts  : {len(web3.eth.accounts)} available")
else:
    print("❌ Connection failed. Ensure Ganache is running.")
    print("   Was your connection successful? NO")

  SETTING UP THE ENVIRONMENT
✅ Connected to Ganache successfully!
   Was your connection successful? YES
   Chain ID  : 1337
   Block #   : 101
   Accounts  : 10 available


---
## SECTION 2: SMART CONTRACT INTERACTION

In [19]:
# ── REPLACE CONTRACT ADDRESS WITH YOUR OWN FROM REMIX ──────────────
contract_address = "0x0D1552fb24282A05AAE767D0A0CFB60578aea2e4"

abi = [
    {
        "inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
        "name": "getRecord",
        "outputs": [
            {"internalType": "string", "name": "", "type": "string"},
            {"internalType": "string", "name": "", "type": "string"},
            {"internalType": "string", "name": "", "type": "string"}
        ],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "getTotalRecords",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "string", "name": "_deviceId", "type": "string"},
            {"internalType": "string", "name": "_dataType",  "type": "string"},
            {"internalType": "string", "name": "_value",     "type": "string"}
        ],
        "name": "storeData",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    }
]
# ───────────────────────────────────────────────────────────────────

contract_address = Web3.to_checksum_address(contract_address)
contract = web3.eth.contract(address=contract_address, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

# Check total records BEFORE storing
total_before = contract.functions.getTotalRecords().call()

print("========================================")
print("  SMART CONTRACT INTERACTION")
print("========================================")
print(f"✅ Smart contract connected!")
print(f"   Smart Contract Address    : {contract_address}")
print(f"   Default Sender Account   : {web3.eth.default_account}")
print(f"   Total Records Before Storing Data: {total_before}")
print(f"   Errors while loading contract? NO")

  SMART CONTRACT INTERACTION
✅ Smart contract connected!
   Smart Contract Address    : 0x0D1552fb24282A05AAE767D0A0CFB60578aea2e4
   Default Sender Account   : 0x35F7DCe12b81724e78945B9548Fee6AE9da4b014
   Total Records Before Storing Data: 0
   Errors while loading contract? NO


---
## SECTION 3: STORING IoT DATA ON THE BLOCKCHAIN
### Part 1 – Load CSV File

In [20]:
import pandas as pd

# Load IoT sensor data from CSV (Homework 1 output)
df = pd.read_csv("iot_data.csv")

# Map columns to smart contract fields
df['device_id']  = 'PKG' + df['package_id'].astype(str)
df['data_type']  = 'Temperature'
df['data_value'] = df['temperature'].round(2).astype(str) + 'C | ' + df['status']

print("========================================")
print("  STORING IoT DATA — CSV PREVIEW")
print("========================================")
print(f"   Number of records in CSV file: {len(df)}")
print()
print("   First 3 records in CSV:")
print(f"   {'#':<5} {'Device ID':<12} {'Data Type':<14} {'Data Value'}")
print("   " + "-" * 55)
for i, row in df[['device_id','data_type','data_value']].head(3).iterrows():
    print(f"   {i+1:<5} {row['device_id']:<12} {row['data_type']:<14} {row['data_value']}")

  STORING IoT DATA — CSV PREVIEW
   Number of records in CSV file: 100

   First 3 records in CSV:
   #     Device ID    Data Type      Data Value
   -------------------------------------------------------
   1     PKG1064      Temperature    3.82C | In Transit
   2     PKG1060      Temperature    2.93C | In Transit
   3     PKG1015      Temperature    5.47C | In Transit


### Part 2 – Store Each Row as a Blockchain Transaction

In [21]:
import time

def send_iot_data(device_id, data_type, data_value):
    """Sends IoT data to the deployed smart contract"""
    txn = contract.functions.storeData(
        device_id, data_type, data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })
    receipt = web3.eth.wait_for_transaction_receipt(txn)
    print(f"✅ Data Stored: {device_id} | {data_type} | {data_value}")
    print(f"   Txn Hash: {receipt.transactionHash.hex()}")
    return receipt

print("========================================")
print("  STORING IoT DATA — TRANSACTIONS")
print("========================================")
print(f"Sending {len(df)} records to blockchain...")
print()

first_txn_hash = None
last_txn_hash  = None
success_count  = 0

for i, (_, row) in enumerate(df.iterrows()):
    receipt = send_iot_data(
        str(row['device_id']),
        str(row['data_type']),
        str(row['data_value'])
    )
    if i == 0:
        first_txn_hash = receipt.transactionHash.hex()
    last_txn_hash = receipt.transactionHash.hex()
    success_count += 1
    time.sleep(0.5)

print()
print("========================================")
print("  TRANSACTION SUMMARY")
print("========================================")
print(f"   Number of records successfully stored : {success_count}")
print(f"   Transaction Hash of FIRST record      : {first_txn_hash}")
print(f"   Transaction Hash of LAST  record      : {last_txn_hash}")

  STORING IoT DATA — TRANSACTIONS
Sending 100 records to blockchain...

✅ Data Stored: PKG1064 | Temperature | 3.82C | In Transit
   Txn Hash: ed2edf7ff43e2fae59e517a1c58ba495c2ed2dd199df66ad0f3d3fa57fc84857
✅ Data Stored: PKG1060 | Temperature | 2.93C | In Transit
   Txn Hash: 57fd79ad4057ca7d5a094fbf2255f80c89060c0bbb6918685d4733e0a5d26200
✅ Data Stored: PKG1015 | Temperature | 5.47C | In Transit
   Txn Hash: 1c8540cb58dda40bf0470b2afb7409dd712fe0065700a741ebd036db08ee6c01
✅ Data Stored: PKG1050 | Temperature | 4.44C | Delayed
   Txn Hash: 46d504fe1e336ea38cc754976b21c60757f0322d4b9e765dda66ca209d2e5698
✅ Data Stored: PKG1048 | Temperature | 2.58C | In Transit
   Txn Hash: 307985d7f1d68a3988f432e7e04947838d4fcaf02a101daf31ed510df8396894
✅ Data Stored: PKG1030 | Temperature | 6.21C | Delayed
   Txn Hash: f0341cb109c6bf10a0e0354c54064a3d4bd215606a263e4867b2d66c2294803c
✅ Data Stored: PKG1025 | Temperature | 7.77C | Delivered
   Txn Hash: 9c4f04881eb2b117e67846c1a08fc87901d05aa671b35f8f

---
## SECTION 4: RETRIEVING & VERIFYING DATA

In [22]:
import datetime

# Get total records after storing
total_after = contract.functions.getTotalRecords().call()

# Get first stored record
first_record = contract.functions.getRecord(0).call()

# Get current timestamp as approximation
latest_block = web3.eth.get_block('latest')
timestamp = datetime.datetime.fromtimestamp(latest_block['timestamp'], datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print("========================================")
print("  RETRIEVING & VERIFYING DATA")
print("========================================")
print(f"   Total records now stored on blockchain: {total_after}")
print()
print("   First stored record retrieved from blockchain:")
print(f"   Timestamp  : {timestamp} UTC")
print(f"   Device ID  : {first_record[0]}")
print(f"   Data Type  : {first_record[1]}")
print(f"   Data Value : {first_record[2]}")
print()

# Display all records as table
all_records = []
for i in range(total_after):
    r = contract.functions.getRecord(i).call()
    all_records.append({
        "Index":      i,
        "Device ID":  r[0],
        "Data Type":  r[1],
        "Data Value": r[2]
    })

result_df = pd.DataFrame(all_records)
print(f"   All {total_after} records retrieved from blockchain:")
result_df

  RETRIEVING & VERIFYING DATA
   Total records now stored on blockchain: 100

   First stored record retrieved from blockchain:
   Timestamp  : 2026-05-30 16:21:12 UTC
   Device ID  : PKG1064
   Data Type  : Temperature
   Data Value : 3.82C | In Transit

   All 100 records retrieved from blockchain:


,Index,Device ID,Data Type,Data Value
0,0,PKG1064,Temperature,3.82C | In Transit
1,1,PKG1060,Temperature,2.93C | In Transit
2,2,PKG1015,Temperature,5.47C | In Transit
3,3,PKG1050,Temperature,4.44C | Delayed
4,4,PKG1048,Temperature,2.58C | In Transit
...,...,...,...,...
95,95,PKG1043,Temperature,7.74C | Delayed
96,96,PKG1093,Temperature,7.37C | Delayed
97,97,PKG1023,Temperature,3.93C | Delivered
98,98,PKG1052,Temperature,6.36C | In Transit
